# Prepare a HiTS data upload: plate maps and `toload` folders

This notebook is the operator's tool for **Part C** (build the `toload` folders) and **Part E**
(clean up) of the *HiTS data upload SOP*. Run the cells top to bottom, one at a time, in Jupyter.

**Before you start**
- The Montreal share is mounted. The expected setup is WSL on Windows with `Z:` mounted at `/mnt/z/` (see the WSL setup doc); `/Volumes/Shared/` on a Mac also works. If the share is mounted anywhere else, set `SHARE_ROOT` in the settings cell. The settings cell checks the share and your paths, and stops with instructions if it can't find them.
- The SSH tunnel to the HiTS database is open, and `mysql_user`, `mysql80_pwd` and `mysql80_port` are set in your environment.
- This repo is installed in the Python environment your Jupyter kernel uses (`uv pip install -e .` or `python -m pip install -e .` from the repo root). Rerun that after pulling a change to `pyproject.toml`. The import names are `database_interactions`, `raw_data_parsers` and `mysql_cnx`, not `smdc_data_proc` (that's only the install name).

This notebook only **reads** from HiTS (production `QB3_HiTS`). It writes files only into the upload folder you name below.

In [ ]:
import os
import numpy as np
import pandas as pd

# Installed with `pip install -e .` (or `uv pip install -e .`) from the repo root, so these work from any folder.
from database_interactions import plate_maps as pm       # plate maps + toload folders
from database_interactions import check_grid_num as cg   # EnVision barcode / plate-order audit

pd.set_option('display.max_columns', 90)

LOG_LEVEL = 'WARNING'   # 'WARNING' shows only what needs your attention; 'INFO' shows every step (for debugging)
pm.set_log_level(LOG_LEVEL)

## 1. Settings (the only cell you normally edit)

- `INFILE`: the scientist's upload workbook (Info tab + plate-map tabs).
- `OUTDIR`: the upload folder on Montreal where the `toload` folders are built.
- `SHARE_ROOT`: leave as `None` unless Montreal is mounted somewhere other than the defaults.
- `USERS`: people who registered the compound lots, used only as a last-resort lot guess.
- `CONTROL_DICT`: the labels the scientist used for positive and negative control wells.
- `REPLACE_DICT`: every text label that appears in the plate maps and is *not* a compound. Map it to `np.nan`.

In [ ]:
# Paths on the Montreal share. Mac: '/Volumes/Shared/...'   Windows: 'Z:\...'   WSL: '/mnt/z/...'
# Keep the r in front of each path: it stops Windows backslashes from being read as escapes,
# so a path copied from File Explorer works as pasted. A Z:\ path also works in WSL (read as /mnt/z/).
# OUTDIR must not end in a backslash: r'...\' is a Python syntax error. Leave the last \ off.
# Windows "Copy as path" adds its own double quotes. Keep exactly one pair around each path,
# matching at both ends: r'...' or r"...", never r'"..."' or r'...".
INFILE = r'/Volumes/Shared/SMDC-users/<scientist>/HITS Upload/toload/<YYYYMMDD>/<workbook>.xlsx'
OUTDIR = r'/Volumes/Shared/SMDC-users/<scientist>/HITS Upload/toload'   # no trailing \

SHARE_ROOT = None                    # None = /mnt/z/ in WSL (or /Volumes/Shared/ on a Mac); else e.g. '/mnt/y/'

USERS = [("First", "Last")]          # whoever registered the compound lots

DOSE_RESPONSE = True                 # True for DR uploads

# Every word in a plate map's ID block (block 0) must be either a control label in CONTROL_DICT
# or a label in REPLACE_DICT (DMSO, Water and Background are built in). Anything else is a stray
# label: remove it from the plate map before running step 2.
CONTROL_DICT = {
    "Sci_PosControl": "No Enzyme",
    "Sci_NegControl": "DMSO",
}
REPLACE_DICT = {"No Enzyme": np.nan, "DMSO": np.nan}

# Map the paths to this machine's share mount, and check they exist. Every later cell uses these.
SHARE_ROOT = SHARE_ROOT or pm.default_share_root()   # stops here if the share isn't mounted
INFILE = pm.share_path(INFILE, SHARE_ROOT, 'INFILE')
OUTDIR = pm.share_path(OUTDIR, SHARE_ROOT, 'OUTDIR')
print('SHARE_ROOT:', SHARE_ROOT)
print('INFILE:    ', INFILE)
print('OUTDIR:    ', OUTDIR)

## 2. Build the plate maps and `toload` folders (SOP Part C, steps 2 to 4)

This writes `<OUTDIR>/<assay id>_<assay name>/toload_<n>/<YYYYMMDD>/`, containing `<grid_num>_DR_plate.xlsx`,
`grid_num.txt` and a copy of the raw file.

An unknown text label stops the run with a `ValueError` that names the label. Add it to `REPLACE_DICT` and run again.

**Read the log for these warnings:**
- *"listed more than once in the Info tab"* or *"already exist in HiTS inventory"*: a destination plate barcode (Grid_num) is repeated or was used before. Fine for a rerun of an earlier upload; otherwise give the plate a new Grid_num.
- *"Date_of_screen more than 92 days old or in the future"*: lists the assay versions and their dates. If it's historical data, carry on. If not, fix the date in the Info tab (often the year) and run step 2 again.
- *"Was a compound pulled from two plates?"*: run step 4 below.
- *"doesn't look like an untouched EnVision export"*: a copied raw CSV was hand-edited in a way Pipeline Pilot rejects (values pasted outside the plate grid, or reads covering different wells). Fix that copy in the toload folder, or rerun the tidy cell below to see the problems again.
- *"Getting lot info from user"*: a lot was guessed as the newest one that user registered. Check those wells by hand.

In [ ]:
info = pm.make_dr_platemaps_and_folders(
    infile=INFILE, users=USERS, basedir=OUTDIR,
    dose_response=DOSE_RESPONSE,
    control_dict=CONTROL_DICT,
    replace_dict=REPLACE_DICT,
    share_root=SHARE_ROOT,
)
info    # the workbook's Info tab, one row per destination plate

## 3. EnVision check: do the barcodes and plate order match `grid_num.txt`? (SOP Part C, step 5)

**Always run this for EnVision data.** Recent uploads loaded results onto the wrong plates because
`grid_num.txt` didn't match the plates in the instrument export.

The check compares the `grid_num.txt` in each folder step 2 just built (not older assays in `OUTDIR`) with the
plates (and their barcodes) in the EnVision CSVs next to it. Run step 2 first in the same session, since the list
of folders comes from its `info`. It writes `folder_summary.csv` and `folder_plate_details.csv` into the deepest
folder those have in common (the assay folder when there is only one assay).

| Verdict | Meaning | What to do |
| --- | --- | --- |
| `OK` | Count and order match, and every barcode matches its grid entry exactly (ignoring case and punctuation) | Go on |
| `ORDER_MISMATCH` | Same plates, different order | Fix the order in `grid_num.txt` (sort by name, not date) and rerun |
| `BARCODES_BLANK` | The export has no barcodes, so the order can't be confirmed | Confirm the plate order with the scientist before uploading |
| `NO_MATCH` / `REVIEW` / `NO_PLATE_BLOCKS` | Count mismatch, unreadable export, or a barcode that only partly matches (e.g. `RUN2026-P01` vs `RUN2026-P99`) | Don't upload. Check the folder by hand |

The number and order of plates in `grid_num.txt` must match the export. Write `BLANK` for an empty plate.

In [ ]:
# Only the folders step 2 just built, not older assays already in OUTDIR
LOAD_FOLDERS = pm.load_folders(info)
print('\n'.join(LOAD_FOLDERS))

problems = cg.check_upload_folder(LOAD_FOLDERS)
problems

## 4. Find compounds pulled from two source plates (only if step 2 warned)

This lists every compound that appears with more than one source plate or well in the `*_DR_plate.xlsx` files
for the assay IDs you name.

In [ ]:
ASSAY_IDS = ['<assay id>']    # e.g. ['2863', '2865']

dups = pm.find_duplicate_source_plates(OUTDIR, ASSAY_IDS)
print(f'{len(dups)} duplicated compound(s)')
dups

### 4b. Fix them (edits the `*_DR_plate.xlsx` files)

For each row above, decide which source plate and well is **right**. `KEEP` gives, for each row of `dups` in order,
the position (0, 1, ...) of the right entry in its `SPlateWell` list. The other entries are removed.

Nothing is written until you set `WRITE = True`.

In [ ]:
KEEP = [0] * len(dups)    # edit: position of the RIGHT source plate/well for each row
WRITE = False

pm.remove_duplicate_source_plates(dups, keep=KEEP, write=WRITE)

## 5. Fill in a control compound's lot (only if needed)

Use this when a control compound has no lot in the `*_DR_plate.xlsx` files.

In [ ]:
ASSAY_IDS = ['<assay id>']
CONTROL_LOTS = {1133707: 1}   # {SMDC_ID: lot}
WRITE = False

pm.fill_control_lots(OUTDIR, ASSAY_IDS, CONTROL_LOTS, write=WRITE)

## 6. Check the DR plate files against the input map (SOP Part C, step 6)

Reads every `*_DR_plate.xlsx` step 2 wrote and compares the SMDC ID in each well with block 0 of that plate's tab in
`INFILE`. Numbers and ID-lot cells like `12346-1` should match; text labels (controls, DMSO, anything in `REPLACE_DICT`)
should be empty in the DR file. It also draws each DR file as a plate of SMDC IDs and of concentrations, so you can
compare it by eye with the scientist's tab.

- `summary`: one row per plate. `mismatched wells` should be 0. Nonzero `no Lot`, `no source plate` or `no conc`
  means those compound wells will load without that field: check them before uploading.
- `mismatches`: every well where the DR file and the input map disagree.

Run step 2 first in the same session, since the list of files comes from its `info`.

In [ ]:
summary, mismatches = pm.check_dr_plates(info, draw=True)
display(summary)
mismatches    # empty = every DR file matches the input map

## Optional: tidy EnVision exports before step 3

Use this only when the loader rejects an EnVision file. It tidies **only the raw files step 2 copied into the
`toload` folders** (the ones that will be uploaded), in place. The originals on the share are not touched.
- Trim extra columns a user added after column 24.
- Make sure each calculated-results block has 3 blank lines above it.
- Fill in empty Z' blocks.

All zeros in the first three counts means that file was already tidy. Run step 2 first in the same session.

`problems` lists anything that makes the file look hand-edited, by line: values pasted outside the plate grid or
over the Plate information / Background information lines, and reads that cover different numbers of wells
(`wells per read`, `reads differ`). Tidying can't fix those, and Pipeline Pilot fails on them (Merge Plates:
*"Well property size mismatch"*). Remove the stray values by hand, and leave missing wells blank, not 0.

In [ ]:
import raw_data_parsers as rdp

RAW_FILES = pm.raw_files(info)   # the raw files step 2 copied into the toload folders
print('\n'.join(RAW_FILES))
rdp.tidy_envision_files(RAW_FILES)

## After the Pipeline Pilot upload: move files to `loaded/` (SOP Part E, step 2)

Pass **the same workbook** you used in step 2, because the function recomputes the folder paths from it.
Keep `delete_orig=False`. There is no dry run, and deleted originals can't be recovered.

In [ ]:
pm.clean_up_folders(infile=INFILE, basedir=OUTDIR, delete_orig=False, share_root=SHARE_ROOT)

## Other lookups

### Plate map from a barcode

In [ ]:
barcode = '<plate barcode>'
pm.make_platemaps_from_barcode(barcode=barcode, outdir=None)

### SMDC IDs and lots from a Biomek (plate + well) list

In [ ]:
BIOMEK_FILE = r'<workbook with a Biomek tab>.xlsx'
BIOMEK_FILE = pm.share_path(BIOMEK_FILE, SHARE_ROOT, 'BIOMEK_FILE')
biomek = pd.read_excel(BIOMEK_FILE, sheet_name='Biomek')
biomek = biomek[['plate', 'well']].rename(columns={'plate': 'Source plate', 'well': 'Source well'})

idlots = pm.lookup_id_and_lot_by_plate_and_well(plates=biomek['Source plate'].tolist(),
                                                 wells=biomek['Source well'].tolist())
biomek.merge(idlots, how='left')